In [0]:
%run ./01-streaming-word-count

In [0]:
class batchWCTestSuite():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("create catalog if not exists spark_stream")
        spark.sql("drop table if exists spark_stream.default.word_count_table")

        dbutils.fs.rm(f"{self.base_data_dir}/checkpoint", True)
        dbutils.fs.rm(f"{self.base_data_dir}/data/text", True)

        dbutils.fs.mkdirs(f"{self.base_data_dir}/data/text")
        print("Done\n")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end='')
        dbutils.fs.cp(f"{self.base_data_dir}/datasets/text/text_data_{itr}.txt", f"{self.base_data_dir}/data/text/")
        print("Done")

    def assertResult(self, expected_count):
        print(f"\tStarting validation...", end='')
        actual_count = spark.sql('''select 
                                        sum(count) 
                                    from spark_stream.default.word_count_table 
                                    where substr(word, 1, 1) == 's'
                                ''')\
                            .collect()[0][0]
        assert expected_count == actual_count, f"Test failed! actual count is {actual_count}"
        print("Done")

    def runTests(self):
        self.cleanTests()
        wc = batchWC()

        print("Testing first iteration of batch word count...") 
        self.ingestData(1)
        wc.wordCount()
        self.assertResult(25)
        print("First iteration of batch word count completed.\n")

        print("Testing second iteration of batch word count...") 
        self.ingestData(2)
        wc.wordCount()
        self.assertResult(32)
        print("Second iteration of batch word count completed.\n") 

        print("Testing third iteration of batch word count...") 
        self.ingestData(3)
        wc.wordCount()
        self.assertResult(37)
        print("Third iteration of batch word count completed.\n")

In [0]:
bwcTS = batchWCTestSuite()
bwcTS.runTests()

In [0]:
class streamWCTestSuite():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("drop table if exists spark_stream.default.word_count_table")

        dbutils.fs.rm(f"{self.checkpoint_dir}/word_count",True)
        dbutils.fs.rm(f"{self.base_data_dir}/data/text", True)

        dbutils.fs.mkdirs(f"{self.base_data_dir}/data/text")
        print("Done\n")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end='')
        dbutils.fs.cp(f"{self.base_data_dir}/datasets/text/text_data_{itr}.txt", f"{self.base_data_dir}/data/text/")
        print("Done")

    def assertResult(self, expected_count):
        print(f"\tStarting validation...", end='')
        actual_count = spark.sql('''select 
                                    sum(count) 
                                 from spark_stream.default.word_count_table 
                                 where substr(word, 1, 1) == 's'
                                 ''').collect()[0][0]
        assert expected_count == actual_count, f"Test failed! actual count is {actual_count}"
        print("Done")

    def runTests(self):
        self.cleanTests()

        wc = streamWC()

        print("Testing first iteration of batch word count...")
        self.ingestData(1)

        sQuery = wc.wordCount()
        sQuery.awaitTermination()

        self.assertResult(25)
        print("First iteration completed.\n")

        print("Testing second iteration of batch word count...")
        self.ingestData(2)

        sQuery = wc.wordCount()
        sQuery.awaitTermination()

        self.assertResult(32)
        print("Second iteration completed.\n")

        print("Testing third iteration of batch word count...")
        self.ingestData(3)

        sQuery = wc.wordCount()
        sQuery.awaitTermination()

        self.assertResult(37)
        print("Third iteration completed.\n")

In [0]:
swcTS = streamWCTestSuite()
swcTS.runTests()